# Integrated Field Moments $\mu_n$

This notebook computes the integrated field moments:

$$\mu_n = \left\langle \left[\int_0^{\lambda_f} \Phi_1(\Omega, \lambda')\,\mathrm{d}\lambda' \right]^n \right\rangle_S$$

for $n = 0, 1, \ldots, n_{\max}$.

The observable $\Phi_1^n$ is represented as $n$ copies of $\phi_1$ at distinct spatial points $x_0, x_1, \ldots, x_{n-1}$, each integrated over $[0, \lambda_f]$. The perturbative expansion produces diagrams whose internal (vertex) times are integrated with causal ordering constraints from $R$ propagators, while the external (observable) times are integrated freely over $[0, \lambda_f]$.

In [1]:
import numpy as np
import time
from IPython.display import display, Math
from scipy.integrate import nquad

from sft_wick import (
    Field, Vertex, Action, compute_moment, reset_uid_counter,
    PropagatorModel, PropagatorCache, integrate_moment,
)

## 1. Physical Setup

Same setup as `PDF_sachs.ipynb`: 3-component field with cubic vertex $F_{abc}\,\psi_a\,\phi_b\,\phi_c$.

In [2]:
# 3-component fields
phi = Field('phi', 'physical', n_components=3)
psi = Field('psi', 'response', n_components=3)

# Vertex: F_{abc} psi_a phi_b phi_c
v = Vertex(fields=[psi, phi, phi], coupling='F')
action = Action(vertices=[v])

# Physical F tensor (0-indexed)
F_physical = np.zeros((3, 3, 3))
F_physical[0, 0, 0] = -0.5   # F_{111}
F_physical[0, 1, 1] = -2.0   # F_{122}
F_physical[0, 2, 2] = -2.0   # F_{133}
F_physical[1, 0, 1] = -1.0   # F_{212}
F_physical[2, 0, 2] = -1.0   # F_{313}

# Code coupling: absorb -i from MSR action
F_code = -1j * F_physical

# Propagator model
def R_time(t, t_prime):
    return np.exp(-(t - t_prime))

def kappa2(n1, t1, n2, t2):
    return np.eye(3)

model = PropagatorModel(
    R_time=R_time, kappa2=kappa2,
    n_components=3, iso_R=True, diag_C=True, t_min=0.0,
)
cache = PropagatorCache(model)

# Pre-tabulate C propagator on a grid for fast lookup
t0 = time.time()
cache.precompute_C_table(t_max=1.0, n_grid=80)
print(f"C table pre-computed in {time.time()-t0:.1f}s (80x80 grid)")
print(f"R(1.0, 0.5) = {R_time(1.0, 0.5):.4f}")
print(f"C_diag(0, 0.5, 0.5) = {cache.C_diagonal(0, 0.5, t2=0.5)}")

C table pre-computed in 10.0s (80x80 grid)
R(1.0, 0.5) = 0.6065
C_diag(0, 0.5, 0.5) = [0.15481812 0.15481812 0.15481812]


## 2. Integration Methods

The library provides `integrate_moment()` with two backends:
- **`method='qmc'`** (default): Quasi-Monte Carlo with Sobol sequences. Maps the unit hypercube to the causal integration domain using the time-ordering DAG. Converges as $O(1/N)$ independent of dimension — essential for high-order diagrams (6D+).
- **`method='nquad'`**: Nested adaptive quadrature via `scipy.integrate.nquad`. Very accurate for low-dimensional integrals but scales as $O(N^d)$.

Combined with `precompute_C_table()` (which replaces expensive `dblquad` C evaluations with fast spline lookups), this makes high-order moment computation tractable.

In [3]:
# Quick timing comparison: nquad vs QMC for mu_1 at order 1
reset_uid_counter()
obs_test = [phi('1', 'x_0')]
result_test = compute_moment(
    obs_test, action, order=1,
    response_phase=True, diag_R=True, diag_C=True, iso_R=True,
)

dt_test = result_test.diagram_terms(1)[0]
ig_test = dt_test.build_integrand({'F': F_code})

# nquad
t0 = time.time()
val_nquad, err_nquad = integrate_moment(ig_test, 1.0, cache, method='nquad')
t_nquad = time.time() - t0

# QMC
t0 = time.time()
val_qmc, err_qmc = integrate_moment(ig_test, 1.0, cache, method='qmc', n_samples=2**14)
t_qmc = time.time() - t0

print(f"nquad:  {val_nquad:+.8f} ± {err_nquad:.2e}  ({t_nquad:.3f}s)")
print(f"QMC:    {val_qmc:+.8f} ± {err_qmc:.2e}  ({t_qmc:.3f}s)")
print(f"Diff:   {abs(val_nquad - val_qmc):.2e}")

nquad:  -0.17633433 ± 5.90e-13  (0.006s)
QMC:    -0.17633433 ± 6.42e-07  (0.289s)
Diff:   8.17e-10


## 3. Compute Moments $\mu_n$

For each $n$, the observable is $n$ copies of $\phi_1$ at distinct points:
$\mathcal{O} = \phi_1(x_0)\,\phi_1(x_1)\,\cdots\,\phi_1(x_{n-1})$.

We sum diagram contributions over all perturbative orders.

In [4]:
def compute_moments(phi, psi, action, coupling_values,
                    n_max, perturbative_order, lambda_f,
                    cache, t_min=0.0, direction=0,
                    method='qmc', n_samples=2**14):
    """Compute mu_n for n = 0, 1, ..., n_max.
    
    Returns dict: {n: {order: value}} for detailed breakdown,
    and array of total moments.
    """
    moments = np.zeros(n_max + 1)
    moments[0] = 1.0  # <1> = 1 (MSR normalization)
    breakdown = {0: {0: 1.0}}
    
    for n in range(1, n_max + 1):
        print(f"\n--- Computing mu_{n} ---")
        reset_uid_counter()
        obs = [phi('1', f'x_{k}') for k in range(n)]
        
        result = compute_moment(
            obs, action, order=perturbative_order,
            response_phase=True,
            diag_R=True, diag_C=True, iso_R=True,
        )
        
        mu_n = 0.0
        order_breakdown = {}
        
        for order in range(perturbative_order + 1):
            dts = result.diagram_terms(order)
            if not dts:
                continue
            
            order_total = 0.0
            t0 = time.time()
            for dt in dts:
                ig = dt.build_integrand(coupling_values)
                val, _ = integrate_moment(
                    ig, lambda_f, cache, t_min=t_min, direction=direction,
                    method=method, n_samples=n_samples,
                )
                order_total += val
            elapsed = time.time() - t0
            
            if abs(order_total) > 1e-15:
                order_breakdown[order] = order_total
                mu_n += order_total
                print(f"  Order {order}: {order_total:+.6f}  ({len(dts)} diagrams, {elapsed:.1f}s)")
        
        moments[n] = mu_n
        breakdown[n] = order_breakdown
        print(f"  Total mu_{n} = {mu_n:.6f}")
    
    return moments, breakdown

## 4. Run: $n_{\max} = 3$, perturbative order = 3, $\lambda_f = 1$

With C pre-tabulation and QMC integration, even $\mu_3$ at order 3 (80 diagrams, 6D integrals) completes in under a minute.

In [5]:
t_start = time.time()
moments, breakdown = compute_moments(
    phi, psi, action, {'F': F_code},
    n_max=3, perturbative_order=3, lambda_f=1.0,
    cache=cache, method='qmc', n_samples=2**14,
)
print(f"\nTotal wall time: {time.time() - t_start:.1f}s")


--- Computing mu_1 ---
  Order 1: -0.176334  (1 diagrams, 0.1s)
  Order 3: -0.032554  (4 diagrams, 1.0s)
  Total mu_1 = -0.208889

--- Computing mu_2 ---
  Order 0: +0.135335  (1 diagrams, 0.1s)
  Order 2: +0.091370  (6 diagrams, 1.4s)
  Total mu_2 = 0.226705

--- Computing mu_3 ---
  Order 1: -0.087504  (6 diagrams, 1.3s)
  Order 3: -0.085321  (80 diagrams, 26.7s)
  Total mu_3 = -0.172825

Total wall time: 30.9s


## 5. Results

In [6]:
print("Integrated field moments:")
print("=" * 40)
for n in range(len(moments)):
    print(f"  mu_{n} = {moments[n]:+.6f}")

print("\nOrder-by-order breakdown:")
print("=" * 40)
for n in sorted(breakdown.keys()):
    print(f"\n  mu_{n}:")
    for order in sorted(breakdown[n].keys()):
        print(f"    Order {order}: {breakdown[n][order]:+.6f}")

Integrated field moments:
  mu_0 = +1.000000
  mu_1 = -0.208889
  mu_2 = +0.226705
  mu_3 = -0.172825

Order-by-order breakdown:

  mu_0:
    Order 0: +1.000000

  mu_1:
    Order 1: -0.176334
    Order 3: -0.032554

  mu_2:
    Order 0: +0.135335
    Order 2: +0.091370

  mu_3:
    Order 1: -0.087504
    Order 3: -0.085321


## 6. Cross-check: $\mu_1$ vs $\int_0^{\lambda_f} \langle\phi_1(x)\rangle\,\mathrm{d}x$

$\mu_1$ should equal the integral of the single-point expectation value $\langle\phi_1(x)\rangle$ from the `PDF_sachs` notebook.

In [7]:
# Cross-check: compute mu_1 via nquad for comparison
reset_uid_counter()
obs_check = [phi('1', 'x_0')]
result_check = compute_moment(
    obs_check, action, order=1,
    response_phase=True, diag_R=True, diag_C=True, iso_R=True,
)

mu1_nquad = 0.0
for order in range(2):
    for dt in result_check.diagram_terms(order):
        ig = dt.build_integrand({'F': F_code})
        val, _ = integrate_moment(ig, 1.0, cache, method='nquad')
        mu1_nquad += val

# Extract mu_1 order-1 contribution from QMC run for comparison
mu1_qmc = sum(breakdown[1].get(o, 0.0) for o in range(4))

print(f"mu_1 (QMC, order 3):     {mu1_qmc:+.6f}")
print(f"mu_1 (nquad, order 1):   {mu1_nquad:+.6f}")
print(f"Order-1 agreement:       {abs(breakdown[1].get(1, 0.0) - mu1_nquad):.2e}")

mu_1 (QMC, order 3):     -0.208889
mu_1 (nquad, order 1):   -0.176334
Order-1 agreement:       1.03e-09


## 7. Cumulants from Moments

The cumulants $\kappa_n$ are related to the raw moments $\mu_n = \langle X^n \rangle$ via the recursion:

$$\kappa_1 = \mu_1, \qquad \kappa_n = \mu_n - \sum_{m=1}^{n-1} \binom{n-1}{m-1}\,\kappa_m\,\mu_{n-m} \quad (n \ge 2).$$

Key quantities:
- **Mean**: $\kappa_1 = \mu_1$
- **Variance**: $\kappa_2 = \mu_2 - \mu_1^2$
- **Third cumulant**: $\kappa_3 = \mu_3 - 3\mu_2\mu_1 + 2\mu_1^3$

In [ ]:
from math import comb

def moments_to_cumulants(mu):
    """Convert raw moments mu_0, mu_1, ..., mu_n to cumulants kappa_1, ..., kappa_n.
    
    Uses the recursion: kappa_n = mu_n - sum_{m=1}^{n-1} C(n-1,m-1) kappa_m mu_{n-m}.
    
    Args:
        mu: array of raw moments [mu_0, mu_1, ..., mu_n] with mu_0 = 1.
    
    Returns:
        Array of cumulants [kappa_1, kappa_2, ..., kappa_n].
    """
    n_max = len(mu) - 1
    kappa = np.zeros(n_max + 1)  # kappa[0] unused
    for n in range(1, n_max + 1):
        kappa[n] = mu[n]
        for m in range(1, n):
            kappa[n] -= comb(n - 1, m - 1) * kappa[m] * mu[n - m]
    return kappa[1:]  # return [kappa_1, ..., kappa_n]

cumulants = moments_to_cumulants(moments)

print("Cumulants:")
print("=" * 40)
for n, kn in enumerate(cumulants, 1):
    print(f"  kappa_{n} = {kn:+.6f}")

sigma = np.sqrt(cumulants[1])
skewness = cumulants[2] / sigma**3
print(f"\nDerived quantities:")
print(f"  mean     = {cumulants[0]:+.6f}")
print(f"  variance = {cumulants[1]:+.6f}")
print(f"  sigma    = {sigma:.6f}")
print(f"  skewness = {skewness:+.6f}")

## 8. PDF via Gram-Charlier Expansion

The Gram-Charlier Type A series approximates the PDF of $X = \int_0^{\lambda_f} \Phi_1\,\mathrm{d}\lambda'$ by expanding around a Gaussian with the same mean and variance:

$$p(x) = \frac{\varphi(z)}{\sigma}\left[1 + \sum_{n=3}^{N} \frac{c_n}{n!}\,\mathrm{He}_n(z)\right], \qquad z = \frac{x - \kappa_1}{\sigma},$$

where $\varphi(z) = e^{-z^2/2}/\sqrt{2\pi}$ is the standard Gaussian, $\mathrm{He}_n$ are the probabilist's Hermite polynomials, and the coefficients $c_n$ are determined by the **standardized cumulants**:

$$c_n = \frac{\kappa_n}{\sigma^n}.$$

For our truncation at $n_{\max} = 3$:

$$p(x) \approx \frac{\varphi(z)}{\sigma}\left[1 + \frac{\gamma_1}{6}\,\mathrm{He}_3(z)\right],$$

with skewness $\gamma_1 = \kappa_3/\sigma^3$ and $\mathrm{He}_3(z) = z^3 - 3z$.

In [ ]:
import matplotlib.pyplot as plt
from scipy.special import factorial

def hermite_he(n, z):
    """Probabilist's Hermite polynomial He_n(z) via recurrence."""
    if n == 0:
        return np.ones_like(z)
    elif n == 1:
        return z
    he_prev, he_curr = np.ones_like(z), z
    for k in range(2, n + 1):
        he_prev, he_curr = he_curr, z * he_curr - (k - 1) * he_prev
    return he_curr

def gram_charlier_pdf(x, cumulants):
    """Gram-Charlier Type A PDF from cumulants [kappa_1, kappa_2, ..., kappa_n].
    
    Args:
        x: array of evaluation points.
        cumulants: array [kappa_1, kappa_2, ..., kappa_n].
    
    Returns:
        PDF values at x.
    """
    mean = cumulants[0]
    sigma = np.sqrt(cumulants[1])
    z = (x - mean) / sigma
    
    # Standard Gaussian
    phi_z = np.exp(-0.5 * z**2) / np.sqrt(2 * np.pi)
    
    # Gram-Charlier correction: sum over n >= 3
    correction = np.ones_like(z)
    for n in range(3, len(cumulants) + 1):
        c_n = cumulants[n - 1] / sigma**n  # standardized cumulant
        correction += (c_n / factorial(n, exact=True)) * hermite_he(n, z)
    
    return phi_z * correction / sigma

# Evaluate PDF
mean = cumulants[0]
x = np.linspace(mean - 4 * sigma, mean + 4 * sigma, 300)

pdf_gc = gram_charlier_pdf(x, cumulants)
pdf_gauss = np.exp(-0.5 * ((x - mean) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(x, pdf_gauss, 'k--', lw=1.5, label='Gaussian')
ax1.plot(x, pdf_gc, 'C0-', lw=2, label=f'Gram-Charlier (order {len(cumulants)})')
ax1.fill_between(x, pdf_gc, alpha=0.15)
ax1.set_xlabel(r'$X = \int_0^{\lambda_f} \Phi_1 \, d\lambda$')
ax1.set_ylabel(r'$p(X)$')
ax1.set_title('PDF of integrated field')
ax1.legend()
ax1.set_xlim(x[0], x[-1])

# Difference plot
ax2.plot(x, pdf_gc - pdf_gauss, 'C3-', lw=2)
ax2.axhline(0, color='k', ls='--', lw=0.8)
ax2.set_xlabel(r'$X$')
ax2.set_ylabel(r'$p_{\mathrm{GC}}(X) - p_{\mathrm{Gauss}}(X)$')
ax2.set_title(f'Non-Gaussian correction (skewness = {skewness:+.3f})')
ax2.set_xlim(x[0], x[-1])

fig.tight_layout()
plt.show()

print(f"\nGram-Charlier PDF integrates to: {np.trapezoid(pdf_gc, x):.6f}")